# 02. Feature Engineering & Data Leakage Audit

This notebook demonstrates the behavioral feature extraction pipeline:
- Velocity windows (1h, 24h, 7d counts, spike ratio)
- Amount Z-score and historical ratios
- Category entropy and novelty flags
- Active hour deviations and geographic distance jumps
- Explicit verification that NO future data or injected labels leak into feature matrix $X$.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import load_featured_data

df_feat = load_featured_data()
print(f"Loaded feature dataset: {df_feat.shape[0]} rows, {df_feat.shape[1]} columns.")

## 1. Inspect Engineered Feature Distributions

In [ ]:
feature_cols = [
    'txns_last_1h', 'txns_last_24h', 'velocity_spike_ratio',
    'amount_zscore', 'amount_to_user_avg_ratio', 'merchant_category_entropy',
    'active_hour_deviation', 'distance_from_home_km'
]
df_feat[feature_cols].describe().round(3)

## 2. Feature Correlation Matrix

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_feat[feature_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Engineered Features')
plt.tight_layout()
plt.show()

## 3. Data Leakage Verification Check
Verifying that `synthetic_anomaly_truth` and `anomaly_type` are completely isolated from feature matrix columns.

In [ ]:
leakage_cols = ['synthetic_anomaly_truth', 'anomaly_type']
found_leakage = [col for col in leakage_cols if col in feature_cols]
if not found_leakage:
    print("PASSED: No data leakage detected! Evaluation labels are isolated.")
else:
    print(f"WARNING: Leakage detected in columns: {found_leakage}")